> **Portfolio summary:** Do optimised mean-variance portfolios beat 1/N out of sample? (group project)
>
> [Project README](README.md) · [Back to the portfolio](../README.md)

# Out-of-Sample Performance of Mean-Variance Portfolio Optimization

**Objective:** Test the out-of-sample performance of the mean-variance model when input parameters are estimated using historic data, and evaluate it against a naïve 1/N portfolio.

**Stocks:** Apple (AAPL), Microsoft (MSFT), NVIDIA (NVDA), JPMorgan (JPM), Johnson & Johnson (JNJ)

**Strategies:**
1. MV optimal (sample estimators, with shorting)
2. MV optimal (sample estimators, no shorting)
3. MV optimal (ARMA/GARCH, with shorting)
4. MV optimal (ARMA/GARCH, no shorting)
5. Naïve 1/N portfolio

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from scipy import optimize as sco
from arch import arch_model
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings('ignore')

## 1. Data Download and Preprocessing

In [ ]:
stock_symbols = ["AAPL", "MSFT", "NVDA", "JPM", "JNJ"]
start_date = "2015-01-01"
end_date = "2025-12-31"

data = yf.download(stock_symbols, start=start_date, end=end_date)
prices = data.loc[:, "Close"]
prices = prices.dropna()
print(f"Data shape: {prices.shape}")
print(f"Date range: {prices.index[0].date()} to {prices.index[-1].date()}")
prices.tail()

In [ ]:
# Calculate daily log returns
returns = prices.pct_change().dropna()
print(f"Returns shape: {returns.shape}")
print(f"\nAnnualized mean returns:")
print(returns.mean() * 252)
print(f"\nAnnualized volatility:")
print(returns.std() * np.sqrt(252))

In [ ]:
# Normalized price plot
nprice = prices / prices.iloc[0]
fig, ax = plt.subplots(figsize=(12, 6))
nprice.plot(ax=ax)
ax.set_title('Normalized Stock Prices (Base = 1)')
ax.set_ylabel('Normalized Price')
ax.set_xlabel('Date')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('normalized_prices.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Portfolio Optimization Functions

In [ ]:
def portfolio_return(weights, mean_returns):
    return np.dot(weights, mean_returns)

def portfolio_variance(weights, cov_matrix):
    return np.dot(weights.T, np.dot(cov_matrix, weights))

def portfolio_std(weights, cov_matrix):
    return np.sqrt(portfolio_variance(weights, cov_matrix))

def optimize_mv_portfolio(mean_returns, cov_matrix, target_return, allow_shorting=True):
    """Solve the Markowitz mean-variance optimization problem."""
    n = len(mean_returns)
    
    def objective(w):
        return portfolio_variance(w, cov_matrix)
    
    constraints = [
        {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
        {'type': 'eq', 'fun': lambda w: portfolio_return(w, mean_returns) - target_return}
    ]
    
    bounds = None if allow_shorting else tuple((0, 1) for _ in range(n))
    
    w0 = np.ones(n) / n
    result = sco.minimize(objective, w0, method='SLSQP', 
                          constraints=constraints, bounds=bounds,
                          options={'ftol': 1e-12, 'maxiter': 1000})
    
    if result.success:
        return result.x
    else:
        return w0  # fallback to equal weight if optimization fails

## 3. Rolling Window Backtest — Sample Estimators

In [ ]:
# Parameters
M = 252  # estimation window: 1 year of trading days
target_return_daily = 0.12 / 252  # target: 12% annualized
N = len(stock_symbols)

# Storage for out-of-sample returns
returns_mv_short = []
returns_mv_noshort = []
returns_naive = []
dates_oos = []

for t in range(M, len(returns) - 1):
    # Estimation window
    est_returns = returns.iloc[t - M:t]
    
    # Unbiased sample estimators
    mu_hat = est_returns.mean().values
    cov_hat = est_returns.cov().values  # pandas .cov() uses N-1 denominator (unbiased)
    
    # MV optimization with shorting
    w_short = optimize_mv_portfolio(mu_hat, cov_hat, target_return_daily, allow_shorting=True)
    
    # MV optimization without shorting
    w_noshort = optimize_mv_portfolio(mu_hat, cov_hat, target_return_daily, allow_shorting=False)
    
    # 1/N weights
    w_naive = np.ones(N) / N
    
    # Out-of-sample return at t+1
    r_next = returns.iloc[t + 1].values
    
    returns_mv_short.append(np.dot(w_short, r_next))
    returns_mv_noshort.append(np.dot(w_noshort, r_next))
    returns_naive.append(np.dot(w_naive, r_next))
    dates_oos.append(returns.index[t + 1])

returns_mv_short = np.array(returns_mv_short)
returns_mv_noshort = np.array(returns_mv_noshort)
returns_naive = np.array(returns_naive)

print(f"Out-of-sample period: {dates_oos[0].date()} to {dates_oos[-1].date()}")
print(f"Number of out-of-sample observations: {len(dates_oos)}")

## 4. Rolling Window Backtest — ARMA/GARCH Estimators

We use ARMA(1,1) for the conditional mean and GARCH(1,1) for the conditional variance of each stock individually. Correlations are estimated using the sample estimator from the rolling window.

In [ ]:
returns_garch_short = []
returns_garch_noshort = []

rebalance_freq = 21  # rebalance monthly to reduce computation time
cached_w_garch_short = np.ones(N) / N
cached_w_garch_noshort = np.ones(N) / N

for idx, t in enumerate(range(M, len(returns) - 1)):
    est_returns = returns.iloc[t - M:t]
    r_next = returns.iloc[t + 1].values
    
    if idx % rebalance_freq == 0:
        mu_garch = np.zeros(N)
        sigma_garch = np.zeros(N)
        
        for i in range(N):
            stock_ret = est_returns.iloc[:, i].values * 100  # scale for numerical stability
            
            try:
                # ARMA(1,1)-GARCH(1,1)
                am = arch_model(stock_ret, mean='ARX', lags=1, vol='Garch', p=1, q=1, dist='normal')
                res = am.fit(disp='off', show_warning=False)
                forecast = res.forecast(horizon=1)
                mu_garch[i] = forecast.mean.iloc[-1, 0] / 100
                sigma_garch[i] = np.sqrt(forecast.variance.iloc[-1, 0]) / 100
            except:
                mu_garch[i] = est_returns.iloc[:, i].mean()
                sigma_garch[i] = est_returns.iloc[:, i].std()
        
        # Build covariance matrix: GARCH variances + sample correlations
        corr_hat = est_returns.corr().values
        D = np.diag(sigma_garch)
        cov_garch = D @ corr_hat @ D
        
        # Ensure positive semi-definiteness
        eigvals = np.linalg.eigvalsh(cov_garch)
        if np.min(eigvals) < 0:
            cov_garch += (-np.min(eigvals) + 1e-8) * np.eye(N)
        
        cached_w_garch_short = optimize_mv_portfolio(mu_garch, cov_garch, target_return_daily, allow_shorting=True)
        cached_w_garch_noshort = optimize_mv_portfolio(mu_garch, cov_garch, target_return_daily, allow_shorting=False)
    
    returns_garch_short.append(np.dot(cached_w_garch_short, r_next))
    returns_garch_noshort.append(np.dot(cached_w_garch_noshort, r_next))
    
    if idx % 500 == 0:
        print(f"Progress: {idx}/{len(returns) - M - 1}")

returns_garch_short = np.array(returns_garch_short)
returns_garch_noshort = np.array(returns_garch_noshort)
print("ARMA/GARCH backtest complete.")

## 5. Performance Summary

In [ ]:
strategies = {
    'MV (Sample, Short)': returns_mv_short,
    'MV (Sample, No Short)': returns_mv_noshort,
    'MV (GARCH, Short)': returns_garch_short,
    'MV (GARCH, No Short)': returns_garch_noshort,
    'Naïve 1/N': returns_naive
}

summary = pd.DataFrame(index=strategies.keys(),
                        columns=['Ann. Mean Return', 'Ann. Std Dev', 'Sharpe Ratio', 'Min Return', 'Max Return'])

for name, rets in strategies.items():
    ann_mean = np.mean(rets) * 252
    ann_std = np.std(rets) * np.sqrt(252)
    sharpe = ann_mean / ann_std if ann_std > 0 else 0
    summary.loc[name] = [f"{ann_mean:.4f}", f"{ann_std:.4f}", f"{sharpe:.4f}", 
                         f"{np.min(rets):.4f}", f"{np.max(rets):.4f}"]

print("\nOut-of-Sample Performance Summary")
print("=" * 80)
print(summary.to_string())

## 6. Mean-Standard Deviation Diagram

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

markers = ['o', 's', '^', 'D', '*']
colors = ['#e74c3c', '#3498db', '#e67e22', '#2ecc71', '#9b59b6']

for i, (name, rets) in enumerate(strategies.items()):
    ann_mean = np.mean(rets) * 252
    ann_std = np.std(rets) * np.sqrt(252)
    ax.scatter(ann_std, ann_mean, marker=markers[i], color=colors[i], 
               s=200, label=name, zorder=5, edgecolors='black', linewidths=0.5)

# Plot individual stocks for reference
oos_returns = returns.iloc[M+1:]
for j, sym in enumerate(stock_symbols):
    stock_ann_mean = oos_returns[sym].mean() * 252
    stock_ann_std = oos_returns[sym].std() * np.sqrt(252)
    ax.scatter(stock_ann_std, stock_ann_mean, marker='x', color='gray', s=80, zorder=4)
    ax.annotate(sym, (stock_ann_std, stock_ann_mean), textcoords="offset points", 
                xytext=(8, 5), fontsize=9, color='gray')

ax.set_xlabel('Annualized Standard Deviation', fontsize=12)
ax.set_ylabel('Annualized Mean Return', fontsize=12)
ax.set_title('Out-of-Sample Mean–Standard Deviation Diagram', fontsize=14)
ax.legend(fontsize=10, loc='best')
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='black', linewidth=0.5, linestyle='--')
plt.tight_layout()
plt.savefig('mean_std_diagram.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Cumulative Wealth Paths

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for i, (name, rets) in enumerate(strategies.items()):
    cumulative = np.cumprod(1 + rets)
    ax.plot(dates_oos, cumulative, label=name, color=colors[i], linewidth=1.5)

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Cumulative Wealth (£1 initial)', fontsize=12)
ax.set_title('Out-of-Sample Cumulative Wealth Paths', fontsize=14)
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('cumulative_wealth.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Rolling Sharpe Ratio (252-day)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
roll_window = 252

for i, (name, rets) in enumerate(strategies.items()):
    rolling_mean = pd.Series(rets).rolling(roll_window).mean() * 252
    rolling_std = pd.Series(rets).rolling(roll_window).std() * np.sqrt(252)
    rolling_sharpe = rolling_mean / rolling_std
    ax.plot(dates_oos, rolling_sharpe, label=name, color=colors[i], alpha=0.8)

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Rolling Sharpe Ratio', fontsize=12)
ax.set_title('252-Day Rolling Sharpe Ratio', fontsize=14)
ax.legend(fontsize=9, loc='best')
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='black', linewidth=0.5, linestyle='--')
plt.tight_layout()
plt.savefig('rolling_sharpe.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Discussion and Interpretation

**Key Findings:**

1. **Estimation Error and Mean-Variance Optimization:** The mean-variance framework, while theoretically optimal under the assumptions of known parameters (Markowitz, 1952), is sensitive to estimation error in expected returns. Small errors in estimating the mean can lead to extreme portfolio weights, particularly when shorting is permitted.

2. **Shorting vs. No-Shorting Constraint:** Portfolios with unrestricted shorting tend to exhibit higher volatility out-of-sample due to amplified estimation error in the weights. The no-shorting constraint acts as an implicit regularization, preventing the optimizer from taking extreme positions based on noisy estimates.

3. **ARMA/GARCH vs. Sample Estimators:** Using GARCH(1,1) for variance forecasting captures time-varying volatility, which is a well-documented feature of financial returns. By incorporating conditional heteroskedasticity, the GARCH-based approach may produce more realistic risk estimates, particularly during periods of volatility clustering.

4. **Naïve 1/N Portfolio:** DeMiguel et al. (2009) showed that the 1/N portfolio is surprisingly competitive against optimized portfolios when estimation windows are short relative to the number of assets. The 1/N portfolio avoids estimation error entirely and benefits from natural diversification.

5. **Mean-Standard Deviation Diagram:** The diagram illustrates the risk-return trade-off achieved by each strategy out-of-sample. Unlike the in-sample efficient frontier, out-of-sample results show that optimized portfolios do not necessarily dominate the 1/N portfolio, highlighting the gap between theoretical and realized performance.

**Conclusion:** The results underscore the challenge of parameter estimation in portfolio optimization. While the mean-variance model provides an elegant theoretical framework, its practical implementation requires careful consideration of estimation uncertainty, constraints, and the trade-off between model sophistication and robustness.